In [ ]:
import os
import pandas as pd
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

nasdaq100_ca = pd.read_csv("nasdaq100_CA.csv")
price_change = pd.read_csv("nasdaq100_price_change.csv")

nasdaq100_ca = nasdaq100_ca.merge(price_change[["symbol", "ytd"]], on="symbol", how="inner")

nasdaq100_ca.head()

for company in nasdaq100_ca["symbol"]:
    prompt = f'''Classify company {company} into one of the following sectors. Answer only with the sector name: Technology, Consumer Cyclical, Industrials, Utilities, Healthcare, Communication, Energy, Consumer Defensive, Real Estate, Financial.
'''
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{ "role": "user", "content": prompt}],
        temperature=0.0,
    )
    sector = response.choices[0].message.content
    
    nasdaq100_ca.loc[nasdaq100_ca["symbol"] == company, "Sector"] = sector
    
nasdaq100_ca["Sector"].value_counts()

prompt = f'''Provide summary information about Nasdaq-100 stock performance year to date (YTD) of companies headquartered in CA, recommending the two best sectors and two or more companies per sector.
            Company data: {nasdaq100_ca} 
'''

response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{ "role": "user", "content": prompt}],
        temperature=0.0,
    )

stock_recommendations = response.choices[0].message.content
print(stock_recommendations)
